## Silver — `cno_areas` (áreas declaradas por obra)

**Origem:** `workspace.bronze.cno_areas` → **Destino:** `workspace.silver.cno_areas`

- **Grão:** 1 linha por área declarada de uma obra (uma obra pode ter N áreas — Principal/Complementar por categoria).
- **Transformações:**
  - Mantém **todas as colunas** da bronze.
  - Correção de tipos: `metragem` para `double`; códigos categóricos padronizados como string trim.
  - Validação dos domínios oficiais RFB (valores fora do domínio são inválidos):
    - `categoria` ∈ {0, 1, 2, 3, 4};
    - `destinacao` ∈ {0..6};
    - `tipo_de_obra` ∈ {0, 1, 2};
    - `tipo_de_area` ∈ {P, C};
    - `tipo_de_area_complementar` ∈ {0, 1, 2, 3} quando preenchido (nulo é permitido quando `tipo_de_area = P`, pois não se aplica);
    - `metragem` não nula e >= 0; `cno` não nulo.
- **Relatório de qualidade:** quantidade de inválidos por regra + % sobre o total bronze.
- **Linhagem:** CSV dados.gov.br → `bronze.cno_areas` → limpeza/validação → `silver.cno_areas`.

In [0]:
%run ./_setup_cno

In [0]:
from pyspark.sql import functions as F
from data_pipeline import (
    save_table,
    add_column_comments,
    resumo_invalidos,
    condicao_valida,
    para_double_seguro,
)
from metadata.metadata import (
    SILVER_CNO_AREAS_COMMENTS,
    DOMINIO_CATEGORIA,
    DOMINIO_DESTINACAO,
    DOMINIO_TIPO_DE_OBRA,
    DOMINIO_TIPO_DE_AREA,
    DOMINIO_TIPO_DE_AREA_COMPLEMENTAR,
)

In [0]:
SOURCE_TABLE = "workspace.bronze.cno_areas"
TARGET_TABLE = "workspace.silver.cno_areas"

In [0]:
df_bronze = spark.table(SOURCE_TABLE)
total_bronze = df_bronze.count()
print(f"Bronze: {total_bronze:,} linhas | colunas: {df_bronze.columns}")

# Todas as colunas, com conversão tolerante (para_double_seguro):
# metragem -> double; códigos -> string trim. Valor malformado vira null
# e é contabilizado no relatório, sem erro mesmo com ANSI mode habilitado
df = df_bronze.select(
    F.trim(F.col("cno").cast("string")).alias("cno"),
    F.trim(F.col("categoria").cast("string")).alias("categoria"),
    F.trim(F.col("destinacao").cast("string")).alias("destinacao"),
    F.trim(F.col("tipo_de_obra").cast("string")).alias("tipo_de_obra"),
    F.trim(F.col("tipo_de_area").cast("string")).alias("tipo_de_area"),
    F.trim(F.col("tipo_de_area_complementar").cast("string")).alias("tipo_de_area_complementar"),
    para_double_seguro(F.col("metragem")).alias("metragem"),
)
display(df.limit(5))

In [0]:
REGRAS_INVALIDOS = {
    "cno_nulo_ou_vazio": F.col("cno").isNull() | (F.col("cno") == ""),
    "categoria_fora_do_dominio": ~F.coalesce(F.col("categoria").isin(list(DOMINIO_CATEGORIA)), F.lit(False)),
    "destinacao_fora_do_dominio": ~F.coalesce(F.col("destinacao").isin(list(DOMINIO_DESTINACAO)), F.lit(False)),
    "tipo_de_obra_fora_do_dominio": ~F.coalesce(F.col("tipo_de_obra").isin(list(DOMINIO_TIPO_DE_OBRA)), F.lit(False)),
    "tipo_de_area_fora_do_dominio": ~F.coalesce(F.col("tipo_de_area").isin(list(DOMINIO_TIPO_DE_AREA)), F.lit(False)),
    "tipo_de_area_complementar_fora_do_dominio": (
        F.col("tipo_de_area_complementar").isNotNull()
        & ~F.col("tipo_de_area_complementar").isin(list(DOMINIO_TIPO_DE_AREA_COMPLEMENTAR))
    ),
    "metragem_nula_ou_negativa": F.col("metragem").isNull() | (F.col("metragem") < 0),
}

In [0]:
df_relatorio = resumo_invalidos(spark, df, REGRAS_INVALIDOS)
print(f"Total bronze avaliado: {total_bronze:,}")
display(df_relatorio)

In [0]:
# Mantém apenas registros válidos em todas as regras
df = df.filter(condicao_valida(REGRAS_INVALIDOS))
print(f"Válidos: {df.count():,} de {total_bronze:,}")
display(df.limit(10))

In [0]:
save_table(df, TARGET_TABLE)
add_column_comments(
    spark,
    TARGET_TABLE,
    SILVER_CNO_AREAS_COMMENTS
)
print(f"Tabela {TARGET_TABLE} persistida: {spark.table(TARGET_TABLE).count():,} linhas")

In [0]:
total = spark.table(TARGET_TABLE).count()
distintos = spark.table(TARGET_TABLE).distinct().count()
cnos = spark.table(TARGET_TABLE).select("cno").distinct().count()
print(f"Total: {total:,} | Linhas únicas: {distintos:,} | Obras distintas: {cnos:,}")
display(spark.sql(f"SELECT tipo_de_area, count(*) AS qtd_areas FROM {TARGET_TABLE} GROUP BY tipo_de_area ORDER BY tipo_de_area"))
display(spark.sql(f"SELECT categoria, destinacao, count(*) AS qtd_areas FROM {TARGET_TABLE} GROUP BY categoria, destinacao ORDER BY categoria, destinacao"))
display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))